# S0 — 03.02: Cross-Entropy & GD for Classification

**Module:** 03 — Supervised Classification  
**Prerequisites:** 00.05 (chain rule), 00.06 (Bernoulli distribution), 00.08 (vectorization), 01.05 (gradient descent update rule), 02.03 (gradient descent for regression), 03.01 (logistic regression forward pass, sigmoid, decision boundary)  
**Difficulty:** Introductory / Foundational  
**Estimated time:** 75–95 minutes core (typical ≈ 85 minutes; see 60-minute cut list below)

> **Learner question:** In Lesson 03.01, we evaluated a logistic regression model with hand-picked coefficients $b = -1.5$ and $w = 1.0$. How does an algorithm actually find coefficients from data, what loss function guides it, and how does gradient descent take steps when the output is a probability?

Module 02 used Mean Squared Error (MSE) to train linear regression models. Lesson 03.01 introduced classification: mapping a feature score $z = b + wx$ through the sigmoid function $\sigma(z) = 1/(1 + e^{-z})$ to produce a predicted probability $p \in (0, 1)$, then thresholding at $0.5$ to predict a discrete class label $\hat{y} \in \{0, 1\}$. But in 03.01, the coefficients were handed to us. We evaluated the model; we did not train it.

This lesson closes that loop. We define the standard objective function for binary classification—**binary cross-entropy** (also called log loss)—derive its gradient using the chain rule, and discover a remarkable mathematical cancellation: the derivative of the sigmoid cancels the denominator of the loss derivative, leaving a clean gradient that structurally mirrors linear regression. We then hand-trace gradient descent step-by-step on eight synthetic examples, explore why cross-entropy succeeds where MSE fails, and analyze what happens when data is perfectly linearly separable.

The dataset is synthetic, deterministic, and dimensionless. Its accuracy is arithmetic on this toy table, not evidence about real-world performance or generalization.

---

### The 60-Minute Cut List
If working under constrained time:
1. **Manual Walkthrough (S5):** Follow State 0 and State 1 arithmetic in detail; skim State 2 and use the programmatic table for States 3–10.
2. **Convexity Exhibit (S3):** Review the midpoint violation table; defer recomputing the Hessian grid by hand.
3. **scikit-learn divergence in S8:** Read the qualitative summary callout; defer the numerical objective comparison.

---

### Symbol Dictionary

| Symbol | Meaning | Domain / Pinned Value |
|---|---|---|
| $N$ | Number of training examples | $N = 8$ (pinned) |
| $x_i$ | Scalar feature value for example $i$ | $x \in \{-3, -2, -1, 0, 1, 2, 3, 4\}$ |
| $y_i$ | Ground truth binary class label | $y_i \in \{0, 1\}$ |
| $A$ | Design matrix $[\mathbf{1}, X]$ | Shape $(8, 2)$ |
| $\theta$ | Parameter vector $[b, w]^T$ | $\theta \in \mathbb{R}^2$; initialized at $\theta_0 = [0, 0]^T$ |
| $b$ | Bias (intercept) parameter | $b \in \mathbb{R}$; $b_0 = 0.0$ |
| $w$ | Weight (slope) parameter | $w \in \mathbb{R}$; $w_0 = 0.0$ |
| $z_i$ | Linear score $b + w x_i$ | $z_i \in \mathbb{R}$; $z_i = 0.0$ at State 0 |
| $\sigma(z)$ | Logistic sigmoid function $1 / (1 + e^{-z})$ | $\sigma(z) \in (0, 1)$ |
| $\sigma'(z)$ | Derivative of the sigmoid: $\sigma(z)(1 - \sigma(z))$ | $\sigma'(z) \in (0, 0.25]$ |
| $p_i$ | Model predicted probability $\sigma(z_i)$ | $p_i \in (0, 1)$; $p_i = 0.5$ at State 0 |
| $\hat{y}_i$ | Discrete class prediction $[p_i \ge 0.5]$ | $\hat{y}_i \in \{0, 1\}$; $\hat{y}_i = 1$ at State 0 |
| $\ell_i$ | Per-example binary cross-entropy loss | $\ell_i \ge 0$ |
| $L(\theta)$ | Mean binary cross-entropy loss $\frac{1}{N}\sum \ell_i$ | $L(\theta) \ge 0$; $L_0 = \ln 2 \approx 0.693147$ |
| $\nabla_\theta L$ | Gradient vector $[\nabla_b L, \nabla_w L]^T$ | $\nabla_\theta L \in \mathbb{R}^2$ |
| $\eta$ | Gradient descent learning rate | $\eta = 0.1$ (pinned) |
| Boundary $x^*$ | Decision boundary location $-b / w$ | $z = 0 \implies x^* = -b / w$ |

---

## S1 — Classification Objectives vs. Regression Residuals

In Lesson 02.03, we trained linear regression models using Mean Squared Error (MSE):
$$L_{\text{MSE}}(\theta) = \frac{1}{N} \sum_{i=1}^N (\hat{y}_i - y_i)^2$$

In regression, targets $y_i$ are continuous numbers on the real line, predictions $\hat{y}_i = b + wx_i$ are unbounded, and the error residual $e_i = \hat{y}_i - y_i$ represents physical or geometric distance. Squaring this distance creates a globally convex parabolic bowl with a single, easily found minimum.

Binary classification is fundamentally different:
1. **Targets are discrete labels:** $y_i \in \{0, 1\}$. An example is either in the positive class or it is not.
2. **Predictions are probabilities:** $p_i = \sigma(z_i) \in (0, 1)$. The model outputs a degree of confidence, not a continuous physical quantity.
3. **The error is a discrepancy in belief:** $p_i - y_i \in (-1, 1)$. If the true label is $1$ and the model predicts $p = 0.99$, the discrepancy is small ($-0.01$). If the model predicts $p = 0.01$, it is confidently wrong.

| Property | Linear Regression (02.01–02.03) | Logistic Regression (03.01–03.02) |
|---|---|---|
| Target $y_i$ | Continuous real number ($\mathbb{R}$) | Binary label ($\{0, 1\}$) |
| Model Output | Unbounded linear score $\hat{y}_i = A\theta$ | Squashed probability $p_i = \sigma(A\theta) \in (0, 1)$ |
| Error Signal | Linear residual $e_i = \hat{y}_i - y_i \in \mathbb{R}$ | Probability discrepancy $p_i - y_i \in (-1, 1)$ |
| Natural Objective | Mean Squared Error (MSE) | Binary Cross-Entropy (Log Loss) |
| Loss Geometry | Strictly convex quadratic bowl | Strictly convex in parameters $\theta$ under CE; **non-convex under MSE** |

If $\hat{y}$ is a probability between $0$ and $1$, why can't we simply minimize $\frac{1}{N} \sum (p_i - y_i)^2$? As we will prove in Section S3, composing MSE with the non-linear sigmoid $\sigma(z)$ produces two fatal flaws:
- **Gradient saturation:** The learning signal vanishes when the model makes confident mistakes.
- **Non-convexity:** The loss surface develops local ripples and saddle points where gradient descent can stall.

To optimize classification models, we need a loss function derived from probability theory: **cross-entropy**.

---

## S2 — The Toy Dataset & The Separability Discovery

We work with an eight-point synthetic dataset with a single scalar feature $x$ and binary target $y$:

| $i$ | $x_i$ | True Label $y_i$ | Meaning |
|---:|---:|:---:|---|
| 1 | -3.0 | 0 | Negative class |
| 2 | -2.0 | 0 | Negative class |
| 3 | -1.0 | 0 | Negative class |
| 4 | 0.0 | 0 | Negative class |
| 5 | 1.0 | 0 | Negative class (highest negative feature) |
| 6 | 2.0 | 1 | Positive class (lowest positive feature) |
| 7 | 3.0 | 1 | Positive class |
| 8 | 4.0 | 1 | Positive class |

### The Linear Separability Discovery
Notice an essential structural property of this dataset:
$$\max(x \mid y=0) = 1.0 < \min(x \mid y=1) = 2.0$$

The negative class points occupy $[-3.0, 1.0]$ and the positive class points occupy $[2.0, 4.0]$. The classes **do not overlap**. Any decision boundary $x^* \in (1.0, 2.0)$ achieves $100\%$ training accuracy. The geometric midpoint of this separating margin is:
$$x^* = \frac{1.0 + 2.0}{2} = 1.5$$

Recall from Lesson 03.01 that our hand-picked coefficients were $b = -1.5$ and $w = 1.0$. Setting $z = -1.5 + 1.0 x^* = 0$ gave $x^* = 1.5$—the exact midpoint of this margin!

### The Separability Divergence (Honest Disclosure)
Because this dataset is perfectly linearly separable, **unregularized logistic regression has no finite minimum**:
> To make the cross-entropy loss strictly zero, the model must predict $p_i = 1.0$ for all positive points and $p_i = 0.0$ for all negative points. But the sigmoid function $\sigma(z) = 1/(1 + e^{-z})$ only reaches $1.0$ as $z \to +\infty$ and $0.0$ as $z \to -\infty$.  
> Therefore, gradient descent will drive weights $b$ and $w$ toward infinity ($\|\theta\| \to \infty$) while their ratio $-b/w$ stabilizes at the maximum-margin decision boundary $\approx 1.5$ (Soudry et al., 2018).

This is not a bug in our code or an arithmetic error; it is a fundamental mathematical property of maximum-likelihood estimation on separable data. Finite iterations of gradient descent represent **snapshots along a diverging trajectory**, not a static terminal point.

---

In [1]:
import numpy as np

# Pinned 8-point synthetic dataset
x = np.array([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=np.float64)
y = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0], dtype=np.float64)
N = len(x)

# Design matrix A = [1, x]
A = np.column_stack([np.ones(N, dtype=np.float64), x])

print(f"Dataset shape: N = {N}")
print(f"Features x:    {x}")
print(f"Labels y:      {y.astype(int)}")
print(f"Design matrix A shape: {A.shape}")
print(f"Separating margin: ({np.max(x[y==0])}, {np.min(x[y==1])}) -> midpoint = 1.5")

Dataset shape: N = 8

Features x:    [-3. -2. -1.  0.  1.  2.  3.  4.]

Labels y:      [0 0 0 0 0 1 1 1]

Design matrix A shape: (8, 2)

Separating margin: (1.0, 2.0) -> midpoint = 1.5



## S3 — Intuition: The Bernoulli Likelihood & Why MSE Fails

### Part A: Deriving Cross-Entropy from Bernoulli Likelihood
Where does binary cross-entropy come from? It is not an arbitrary formula; it is the negative log-likelihood of independent coin tosses.

1. **Step 1 (Probability Model):** Model label $y_i \in \{0, 1\}$ as a Bernoulli trial where $P(Y_i = 1 \mid x_i) = p_i$ and $P(Y_i = 0 \mid x_i) = 1 - p_i$. In a single compact expression:
   $$P(Y_i = y_i \mid x_i) = p_i^{y_i} (1 - p_i)^{1 - y_i}$$
   - When $y_i = 1$: $p_i^1 (1 - p_i)^0 = p_i$.
   - When $y_i = 0$: $p_i^0 (1 - p_i)^1 = 1 - p_i$.

2. **Step 2 (Product Likelihood):** Assuming examples are conditionally independent given $x_i$, the likelihood of observing the entire dataset is the product:
   $$\mathcal{L}(\theta) = \prod_{i=1}^N p_i^{y_i} (1 - p_i)^{1 - y_i}$$

3. **Step 3 (Negative Log-Likelihood):** Multiplying probabilities causes numerical underflow in floating-point arithmetic. Taking the natural logarithm converts products into sums:
   $$\ln \mathcal{L}(\theta) = \sum_{i=1}^N \left[ y_i \ln p_i + (1 - y_i) \ln(1 - p_i) \right]$$
   Because the logarithm is strictly monotonic, maximizing likelihood is equivalent to minimizing the **mean negative log-likelihood**:
   $$L(\theta) = -\frac{1}{N} \ln \mathcal{L}(\theta) = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \ln p_i + (1 - y_i) \ln(1 - p_i) \right]$$

This is the **binary cross-entropy loss** (also called log loss).

---

### Part B: The $-\ln(p)$ Penalty Profile
Examining the per-example loss $\ell_i = -\left[ y_i \ln p_i + (1 - y_i) \ln(1 - p_i) \right]$:
- **If $y_i = 1$:** The loss is $-\ln(p_i)$.
  - If $p_i = 1.0$: $-\ln(1.0) = 0$ (zero penalty for confident correct prediction).
  - If $p_i = 0.5$: $-\ln(0.5) = \ln 2 \approx 0.693147$.
  - If $p_i = 0.01$: $-\ln(0.01) \approx 4.605$.
  - As $p_i \to 0$: $-\ln(p_i) \to +\infty$ (infinite penalty for confident wrong prediction).
- **If $y_i = 0$:** The loss is $-\ln(1 - p_i)$.
  - If $p_i = 0.0$: $-\ln(1.0) = 0$.
  - If $p_i = 0.5$: $-\ln(0.5) \approx 0.693147$.
  - As $p_i \to 1$: $-\ln(1 - p_i) \to +\infty$.

The logarithmic curve produces an asymmetrical penalty: small errors incur small penalties, but confident mistakes are punished severely. Because $0 < p_i < 1$, both $\ln(p_i) < 0$ and $\ln(1 - p_i) < 0$, ensuring $L(\theta) \ge 0$ everywhere.

---

### Part C: Why MSE Fails for Classification
Why can't we use Mean Squared Error $J_{\text{MSE}} = \frac{1}{2}(p_i - y_i)^2$?

#### 1. Gradient Saturation
Differentiating MSE with respect to score $z_i$:
$$\frac{\partial J_{\text{MSE}}}{\partial z_i} = \frac{\partial J_{\text{MSE}}}{\partial p_i} \frac{\partial p_i}{\partial z_i} = (p_i - y_i) \cdot \sigma'(z_i) = (p_i - y_i) \cdot p_i(1 - p_i)$$
Suppose $y_i = 1$, but the model is confidently wrong: $z_i = -5.0 \implies p_i \approx 0.00669$. The error residual is large: $p_i - y_i \approx -0.99331$. However:
$$\sigma'(-5.0) = p_i(1 - p_i) = 0.00669 \times 0.99331 \approx 0.00665$$
$$\frac{\partial J_{\text{MSE}}}{\partial z_i} = (-0.99331) \times 0.00665 \approx -0.00660$$
The gradient is virtually zero! Because the sigmoid saturated in the flat tail, the factor $p_i(1 - p_i)$ choked off the error signal. The model is confidently wrong, yet gradient descent takes virtually no step.

#### 2. Non-Convexity Proof
Composing MSE with the sigmoid destroys convexity. On our 8-point dataset, computing the Hessian of MSE over a grid of 32,761 points $(b, w) \in [-4, 4] \times [-4, 4]$ reveals that the Hessian is indefinite (possessing negative eigenvalues) at **25,106 of 32,761 grid points** (76.6% of the grid).

We demonstrate a concrete **midpoint convexity violation** along the line segment between points $a = (4.250, -5.000)$ and $c = (-0.500, 0.750)$ with midpoint $\text{mid} = \frac{a + c}{2} = (1.875, -2.125)$:

| Point | Parameter $(b, w)$ | MSE Loss | Cross-Entropy Loss |
|---|---|---:|---:|
| Endpoint $a$ | $(4.250, -5.000)$ | $0.883553$ | $6.273010$ |
| Endpoint $c$ | $(-0.500, 0.750)$ | $0.078250$ | $3.970434$ |
| **Chord Average** $\frac{L(a) + L(c)}{2}$ | — | **$0.480901$** | **$5.121722$** |
| **Evaluated Midpoint** | $(1.875, -2.125)$ | **$0.839431$** | **$4.323815$** |
| **Convexity Status** | $L(\text{mid}) \le \text{Chord}$? | **VIOLATED ($0.839 > 0.481$)** | **SATISFIED ($4.324 \le 5.122$)** |

For MSE, the loss at the midpoint is nearly double the average of the endpoints. A ball placed on this surface would roll into a local ripple. On the exact same segment, cross-entropy satisfies midpoint convexity.

---

## S4 — Mathematical Derivation: Differentiating Through the Sigmoid

We derive the gradient of binary cross-entropy with respect to parameters $\theta = [b, w]^T$.

### Step 1: Derivative of the Sigmoid Function
Recall the definition of the sigmoid:
$$\sigma(z) = \frac{1}{1 + e^{-z}} = (1 + e^{-z})^{-1}$$

Differentiating with respect to $z$ using the power rule and chain rule:
$$\sigma'(z) = \frac{d}{dz} (1 + e^{-z})^{-1} = -(1 + e^{-z})^{-2} \cdot \frac{d}{dz}(1 + e^{-z}) = -(1 + e^{-z})^{-2} \cdot (-e^{-z}) = \frac{e^{-z}}{(1 + e^{-z})^2}$$

Decomposing the fraction:
$$\sigma'(z) = \frac{1}{1 + e^{-z}} \cdot \frac{e^{-z}}{1 + e^{-z}} = \sigma(z) \cdot \left( \frac{1 + e^{-z} - 1}{1 + e^{-z}} \right) = \sigma(z) \cdot \left( 1 - \frac{1}{1 + e^{-z}} \right) = \sigma(z)(1 - \sigma(z))$$

$$\sigma'(z) = p(1 - p)$$

The derivative of the sigmoid is simply the output probability multiplied by its complement. Its maximum value is $0.25$ (at $z = 0$, where $p = 0.5$).

---

### Step 2: The Three-Link Chain Rule
For a single training example $(x_i, y_i)$, the loss is:
$$\ell_i = -[y_i \ln p_i + (1 - y_i) \ln(1 - p_i)]$$
where $p_i = \sigma(z_i)$ and $z_i = b + wx_i$.

To find $\frac{\partial \ell_i}{\partial w}$, apply the chain rule across three links:
$$\frac{\partial \ell_i}{\partial w} = \frac{\partial \ell_i}{\partial p_i} \cdot \frac{\partial p_i}{\partial z_i} \cdot \frac{\partial z_i}{\partial w}$$

#### Link 1: Derivative of loss with respect to probability
$$\frac{\partial \ell_i}{\partial p_i} = -\left[ \frac{y_i}{p_i} - \frac{1 - y_i}{1 - p_i} \right] = -\left[ \frac{y_i(1 - p_i) - (1 - y_i)p_i}{p_i(1 - p_i)} \right] = -\left[ \frac{y_i - y_i p_i - p_i + y_i p_i}{p_i(1 - p_i)} \right] = -\frac{y_i - p_i}{p_i(1 - p_i)} = \frac{p_i - y_i}{p_i(1 - p_i)}$$

#### Link 2: Derivative of probability with respect to score
$$\frac{\partial p_i}{\partial z_i} = \sigma'(z_i) = p_i(1 - p_i)$$

#### The Great Cancellation
Multiplying Link 1 and Link 2:
$$\frac{\partial \ell_i}{\partial z_i} = \frac{\partial \ell_i}{\partial p_i} \cdot \frac{\partial p_i}{\partial z_i} = \left[ \frac{p_i - y_i}{p_i(1 - p_i)} \right] \cdot \left[ p_i(1 - p_i) \right]$$

The term $p_i(1 - p_i)$ in the denominator cancels the exact same term in the numerator:
$$\frac{\partial \ell_i}{\partial z_i} = p_i - y_i$$

> **The Great Cancellation:** As Goodfellow et al. (*Deep Learning*, §6.2.2.2) observed:  
> *"The log in the cost function undoes the exp of the sigmoid."*  
> When the sigmoid saturates, its derivative $p(1-p)$ becomes tiny. But the derivative of $-\ln p$ contains $1/p$, which blows up at exactly the reciprocal rate. The two effects cancel out, leaving a clean, linear error term $p_i - y_i$.

#### Link 3: Derivative of score with respect to parameters
$$z_i = b + wx_i \implies \frac{\partial z_i}{\partial b} = 1, \quad \frac{\partial z_i}{\partial w} = x_i$$

Combining all three links gives the per-example parameter gradients:
$$\frac{\partial \ell_i}{\partial b} = p_i - y_i$$
$$\frac{\partial \ell_i}{\partial w} = (p_i - y_i)x_i$$

---

### Step 3: Batch Gradient & Matrix Form
Averaging over all $N$ training examples:
$$\nabla_b L = \frac{\partial L}{\partial b} = \frac{1}{N} \sum_{i=1}^N (p_i - y_i)$$
$$\nabla_w L = \frac{\partial L}{\partial w} = \frac{1}{N} \sum_{i=1}^N (p_i - y_i)x_i$$

Let $A = [\mathbf{1}, X] \in \mathbb{R}^{N \times 2}$ be the design matrix, $p = [p_1, \dots, p_N]^T \in \mathbb{R}^N$ be the predicted probabilities, and $y = [y_1, \dots, y_N]^T \in \mathbb{R}^N$ be the true labels. The gradient vector $\nabla_\theta L = [\nabla_b L, \nabla_w L]^T$ is expressed compactly in matrix form:
$$\nabla_\theta L = \frac{1}{N} A^T (p - y)$$

---

### Structural Parallel Box

```text
========================================================================================
STRUCTURAL PARALLEL: LINEAR REGRESSION VS. LOGISTIC REGRESSION
========================================================================================
Linear Regression MSE (02.03):
    grad = (2 / N) * A.T @ (y_hat - y),    where y_hat = A @ theta     (linear map)

Logistic Regression Cross-Entropy (03.02):
    grad = (1 / N) * A.T @ (p - y),        where p = sigma(A @ theta)  (sigmoid map)

Shared architecture: (constant / N) * A.T @ (prediction - target)
1. The factor of 2 in MSE arises from squaring: d/du (u^2) = 2u.
2. In cross-entropy, that constant is 1.
3. The prediction map changes from linear y_hat in R to squashed probability p in (0, 1).
4. The gradient descent update rule is structurally identical:
       theta_next = theta - eta * grad
========================================================================================
```

---

## S5 — Manual Walkthrough & Arithmetic Trace

We trace the optimization trajectory by hand, verifying every decimal.

### State 0: Initial Condition ($\theta_0 = [0, 0]^T$, $\eta = 0.1$)
At initialization $\theta_0 = [0, 0]^T$:
- Score: $z_i = 0 + 0 \cdot x_i = 0.000000$ for all eight examples.
- Probability: $p_i = \sigma(0) = \frac{1}{1 + e^{-0}} = \frac{1}{1 + 1} = 0.500000$ for all eight examples.
- Class prediction: Under the standard rule $\hat{y}_i = [p_i \ge 0.5]$, the tie $0.5 \ge 0.5$ classifies **all eight examples as positive ($\hat{y}_i = 1$)**.
- Accuracy: True labels are five $0$s and three $1$s. With all $\hat{y}_i = 1$, exactly 3 predictions are correct:
  $$\text{Accuracy}_0 = \frac{3}{8} = 0.375$$
- Loss per example: $\ell_i = -[y_i \ln(0.5) + (1 - y_i)\ln(0.5)] = -\ln(0.5) = \ln 2 \approx 0.693147$.
  $$L_0 = \frac{1}{8} \sum_{i=1}^8 \ln 2 = \ln 2 \approx 0.693147$$
- Residuals $p_i - y_i$:
  - For $y=0$ ($i=1..5$): $0.5 - 0 = +0.5$.
  - For $y=1$ ($i=6..8$): $0.5 - 1 = -0.5$.
- Gradients:
  $$\nabla_b L_0 = \frac{1}{8} \sum_{i=1}^8 (p_i - y_i) = \frac{5(+0.5) + 3(-0.5)}{8} = \frac{2.5 - 1.5}{8} = \frac{1.0}{8} = +0.125000$$
  $$\nabla_w L_0 = \frac{1}{8} \sum_{i=1}^8 (p_i - y_i)x_i = \frac{0.5(-3 -2 -1 +0 +1) - 0.5(2 + 3 + 4)}{8} = \frac{0.5(-5) - 0.5(9)}{8} = \frac{-2.5 - 4.5}{8} = \frac{-7.0}{8} = -0.875000$$

### Table T2: Per-Example Audit at State 0 ($\theta_0 = [0, 0]$)
| $i$ | $x_i$ | $y_i$ | $z_i$ | $e^{-z_i}$ | $p_i$ | $p_i - y_i$ | $(p_i - y_i)x_i$ | $\ell_i$ | $\hat{y}_i$ | Correct? |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| 1 | -3.0 | 0 | 0.000000 | 1.000000 | 0.500000 | +0.500000 | -1.500000 | 0.693147 | 1 | No |
| 2 | -2.0 | 0 | 0.000000 | 1.000000 | 0.500000 | +0.500000 | -1.000000 | 0.693147 | 1 | No |
| 3 | -1.0 | 0 | 0.000000 | 1.000000 | 0.500000 | +0.500000 | -0.500000 | 0.693147 | 1 | No |
| 4 | 0.0 | 0 | 0.000000 | 1.000000 | 0.500000 | +0.500000 | 0.000000 | 0.693147 | 1 | No |
| 5 | 1.0 | 0 | 0.000000 | 1.000000 | 0.500000 | +0.500000 | +0.500000 | 0.693147 | 1 | No |
| 6 | 2.0 | 1 | 0.000000 | 1.000000 | 0.500000 | -0.500000 | -1.000000 | 0.693147 | 1 | Yes |
| 7 | 3.0 | 1 | 0.000000 | 1.000000 | 0.500000 | -0.500000 | -1.500000 | 0.693147 | 1 | Yes |
| 8 | 4.0 | 1 | 0.000000 | 1.000000 | 0.500000 | -0.500000 | -2.000000 | 0.693147 | 1 | Yes |
| **Sum** | | | | | | **+1.000000** | **-7.000000** | **5.545177** | | **3 / 8** |
| **Mean** | | | | | | **$\nabla_b = 0.125000$** | **$\nabla_w = -0.875000$** | **$L = 0.693147$** | | **acc = 0.375** |

---

### Update to State 1 ($\eta = 0.1$)
Apply the gradient descent update rule $\theta_1 = \theta_0 - \eta \nabla L_0$:
$$b_1 = 0.000000 - 0.1 \times (0.125000) = -0.012500$$
$$w_1 = 0.000000 - 0.1 \times (-0.875000) = +0.087500$$
$$\theta_1 = [-0.012500, \, 0.087500]^T$$

Where is the new decision boundary? Setting $z = b_1 + w_1 x^* = 0$:
$$x^* = -\frac{b_1}{w_1} = -\frac{-0.012500}{0.087500} = \frac{125}{875} = \frac{1}{7} \approx 0.142857$$
*(Notice: because $\theta_0 = [0, 0]$, this first boundary location $1/7$ is an exact invariant independent of the learning rate $\eta$!)*

### Table T3: Per-Example Audit at State 1 ($\theta_1 = [-0.012500, 0.087500]$)
For each example, $z_i = -0.0125 + 0.0875 x_i$:
| $i$ | $x_i$ | $y_i$ | $z_i$ | $e^{-z_i}$ | $p_i$ | $p_i - y_i$ | $(p_i - y_i)x_i$ | $\ell_i$ | $\hat{y}_i$ | Correct? |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| 1 | -3.0 | 0 | -0.275000 | 1.316531 | 0.431680 | +0.431680 | -1.295040 | 0.565071 | 0 | Yes |
| 2 | -2.0 | 0 | -0.187500 | 1.206230 | 0.453262 | +0.453262 | -0.906524 | 0.603785 | 0 | Yes |
| 3 | -1.0 | 0 | -0.100000 | 1.105171 | 0.475021 | +0.475021 | -0.475021 | 0.644397 | 0 | Yes |
| 4 | 0.0 | 0 | -0.012500 | 1.012578 | 0.496875 | +0.496875 | 0.000000 | 0.686917 | 0 | Yes |
| 5 | 1.0 | 0 | +0.075000 | 0.927743 | 0.518741 | +0.518741 | +0.518741 | 0.731350 | 1 | **No** |
| 6 | 2.0 | 1 | +0.162500 | 0.850016 | 0.540536 | -0.459464 | -0.918928 | 0.615194 | 1 | Yes |
| 7 | 3.0 | 1 | +0.250000 | 0.778801 | 0.562177 | -0.437823 | -1.313470 | 0.575939 | 1 | Yes |
| 8 | 4.0 | 1 | +0.337500 | 0.713552 | 0.583583 | -0.416417 | -1.665668 | 0.538568 | 1 | Yes |
| **Sum** | | | | | | **+1.061872** | **-6.055910** | **4.961221** | | **7 / 8** |
| **Mean** | | | | | | **$\nabla_b = 0.132734$** | **$\nabla_w = -0.756989$** | **$L = 0.620153$** | | **acc = 0.875** |

- **Loss descent verified:** $L_1 = 0.620153 < L_0 = 0.693147$. The loss dropped by $0.072994$.
- **Accuracy jump:** The boundary $x^* = 1/7 \approx 0.143$ now sits between $x = 0$ and $x = 1$. The only misclassified point is $x = 1.0$ (which is $y=0$ but has $z > 0$, so $\hat{y} = 1$). Accuracy jumps from $3/8 \to 7/8 = 0.875$.

---

### Update to State 2 ($\eta = 0.1$)
$$b_2 = -0.012500 - 0.1 \times (0.132734) = -0.025773$$
$$w_2 = +0.087500 - 0.1 \times (-0.756989) = +0.163199$$
$$\theta_2 = [-0.025773, \, 0.163199]^T$$
$$x^* = -\frac{-0.025773}{0.163199} \approx 0.157927$$

### Table T4: Per-Example Audit at State 2 ($\theta_2 = [-0.025773, 0.163199]$)
| $i$ | $x_i$ | $y_i$ | $z_i$ | $e^{-z_i}$ | $p_i$ | $p_i - y_i$ | $(p_i - y_i)x_i$ | $\ell_i$ | $\hat{y}_i$ | Correct? |
|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| 1 | -3.0 | 0 | -0.515370 | 1.674258 | 0.373936 | +0.373936 | -1.121807 | 0.468302 | 0 | Yes |
| 2 | -2.0 | 0 | -0.352171 | 1.422152 | 0.412856 | +0.412856 | -0.825712 | 0.532485 | 0 | Yes |
| 3 | -1.0 | 0 | -0.188972 | 1.208007 | 0.452897 | +0.452897 | -0.452897 | 0.603118 | 0 | Yes |
| 4 | 0.0 | 0 | -0.025773 | 1.026108 | 0.493557 | +0.493557 | 0.000000 | 0.680343 | 0 | Yes |
| 5 | 1.0 | 0 | +0.137425 | 0.871599 | 0.534302 | +0.534302 | +0.534302 | 0.764219 | 1 | **No** |
| 6 | 2.0 | 1 | +0.300624 | 0.740356 | 0.574595 | -0.425405 | -0.850810 | 0.554090 | 1 | Yes |
| 7 | 3.0 | 1 | +0.463823 | 0.628875 | 0.613921 | -0.386079 | -1.158238 | 0.487889 | 1 | Yes |
| 8 | 4.0 | 1 | +0.627022 | 0.534180 | 0.651814 | -0.348186 | -1.392744 | 0.427996 | 1 | Yes |
| **Sum** | | | | | | **+1.107877** | **-5.267905** | **4.518442** | | **7 / 8** |
| **Mean** | | | | | | **$\nabla_b = 0.138485$** | **$\nabla_w = -0.658488$** | **$L = 0.564805$** | | **acc = 0.875** |

- **Loss descent verified:** $L_2 = 0.564805 < L_1 = 0.620153$. Loss fell by $0.055348$.
- **Accuracy invariant:** The boundary moved slightly ($0.143 \to 0.158$), but remains in $(0, 1)$. Example $x=1$ is still the lone error. Accuracy stays flat at $7/8 = 0.875$.

---

### Table T1: Complete Trace for States 0 through 10
The remaining states follow the identical algorithm:

| State $k$ | $b$ | $w$ | Boundary $-b/w$ | Loss $L(\theta)$ | $\nabla_b L$ | $\nabla_w L$ | Accuracy |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 0.000000 | 0.000000 | — | 0.693147 | 0.125000 | -0.875000 | 0.375 (3/8) |
| 1 | -0.012500 | 0.087500 | 0.142857 | 0.620153 | 0.132734 | -0.756989 | 0.875 (7/8) |
| 2 | -0.025773 | 0.163199 | 0.157927 | 0.564805 | 0.138485 | -0.658488 | 0.875 (7/8) |
| 3 | -0.039622 | 0.229048 | 0.172985 | 0.522192 | 0.142526 | -0.577482 | 0.875 (7/8) |
| 4 | -0.053874 | 0.286796 | 0.187850 | 0.488733 | 0.145227 | -0.510994 | 0.875 (7/8) |
| 5 | -0.068397 | 0.337895 | 0.202421 | 0.461917 | 0.146923 | -0.456149 | 0.875 (7/8) |
| 6 | -0.083089 | 0.383510 | 0.216655 | 0.439996 | 0.147879 | -0.410525 | 0.875 (7/8) |
| 7 | -0.097877 | 0.424563 | 0.230537 | 0.421748 | 0.148294 | -0.372205 | 0.875 (7/8) |
| 8 | -0.112707 | 0.461783 | 0.244069 | 0.406306 | 0.148312 | -0.339702 | 0.875 (7/8) |
| 9 | -0.127538 | 0.495753 | 0.257261 | 0.393046 | 0.148038 | -0.311870 | 0.875 (7/8) |
| 10 | -0.142342 | 0.526940 | 0.270129 | 0.381510 | 0.147547 | -0.287826 | 0.875 (7/8) |

> **Critical Pedagogical Takeaway:** Look closely at States 1 through 10. The loss drops continuously from $0.620153 \to 0.381510$ (a $38\%$ reduction). Yet accuracy **does not change at all**—it remains locked at $0.875$ ($7/8$).  
> This illustrates why loss is the training objective and accuracy is merely an evaluation metric. Gradient descent does not optimize accuracy; it optimizes cross-entropy, pushing predicted probabilities toward higher confidence even when discrete classifications remain identical.

---

### Long-Run Milestones (Table T5)
What happens if we continue running gradient descent?

| Milestone $k$ | $b$ | $w$ | Boundary $-b/w$ | Loss $L(\theta)$ | Accuracy | Significance |
|---:|---:|---:|---:|---:|---:|---|
| **$k=0$** | 0.000000 | 0.000000 | — | 0.693147 | 0.375 | Initial state (tie threshold classifies all as 1) |
| **$k=1$** | -0.012500 | 0.087500 | 0.142857 | 0.620153 | 0.875 | Boundary jumps into interval $(0, 1)$ |
| **$k=10$** | -0.142342 | 0.526940 | 0.270129 | 0.381510 | 0.875 | End of manual trace; steady probability refinement |
| **$k=25$** | -0.344400 | 0.822839 | 0.418551 | 0.292154 | 0.875 | Boundary continues migrating rightward |
| **$k=50$** | -0.630040 | 1.077226 | 0.584873 | 0.237255 | 0.875 | Loss cuts below $0.25$ |
| **$k^*=139$** | **-1.389114** | **1.388957** | **1.000113** | **0.161692** | **1.000** | **First all-correct state! Boundary crosses $x=1.0$** |
| **$k=500$** | -2.709971 | 2.195726 | 1.234203 | 0.089851 | 1.000 | Boundary settles deep inside margin $(1, 2)$ |
| **$k=5,000$** | **-7.004738** | **4.844064** | **1.446046** | **0.022079** | **1.000** | Visual display cap; boundary near midpoint $1.5$ |
| **$k=2,000,000$** | -24.607956 | 16.562013 | 1.485807 | 0.000065 | 1.000 | Weights continue diverging; loss never hits zero |

---

### Snapshot Comparison (Table T6)
Comparing different checkpoints and models in the literature:

| Snapshot / Model | $b$ | $w$ | Boundary $-b/w$ | Mean Loss | Note / Provenance |
|---|---:|---:|---:|---:|---|
| **First all-correct ($k^*=139$)** | -1.389114 | 1.388957 | 1.000113 | 0.161692 | Boundary just clears $x=1.0$; $100\%$ accuracy |
| **Lesson 03.01 coefficients** | -1.500000 | 1.000000 | 1.500000 | 0.193695 | Hand-picked values; exact midpoint boundary |
| **scikit-learn default ($C=1.0$)** | -1.760949 | 1.155912 | 1.523428 | 0.168337 | L2-penalized sum-form solution |
| **Visual display cap ($k=5,000$)** | -7.004738 | 4.844064 | 1.446046 | 0.022079 | Our GD trace; sharp sigmoid separator |

---

In [2]:
# S6 — Teaching Implementation: Full-Batch Logistic GD
import numpy as np

def sigmoid(z: np.ndarray) -> np.ndarray:
    """Vectorized logistic sigmoid: sigma(z) = 1 / (1 + exp(-z))."""
    return 1.0 / (1.0 + np.exp(-z))

def cross_entropy_loss(y: np.ndarray, p: np.ndarray, eps: float = 1e-15) -> float:
    """Mean binary cross-entropy loss with numerical guardrail against log(0)."""
    p_clipped = np.clip(p, eps, 1.0 - eps)
    return float(-np.mean(y * np.log(p_clipped) + (1.0 - y) * np.log(1.0 - p_clipped)))

def cross_entropy_gradient(A: np.ndarray, y: np.ndarray, p: np.ndarray) -> np.ndarray:
    """Batch gradient: (1 / N) * A.T @ (p - y)."""
    N = A.shape[0]
    return (1.0 / N) * (A.T @ (p - y))

def train_logistic_gd(
    A: np.ndarray,
    y: np.ndarray,
    eta: float,
    n_iterations: int,
    theta0: np.ndarray = None
) -> dict:
    """Train logistic regression with full-batch gradient descent."""
    A_arr = np.asarray(A, dtype=np.float64)
    y_arr = np.asarray(y, dtype=np.float64)
    
    # Input validation
    if A_arr.ndim != 2 or A_arr.shape[1] != 2:
        raise ValueError("Design matrix A must have shape (N, 2)")
    if y_arr.ndim != 1 or y_arr.shape[0] != A_arr.shape[0]:
        raise ValueError("Target y must have shape (N,) matching A")
    if not np.all((y_arr == 0.0) | (y_arr == 1.0)):
        raise ValueError("Target y must contain only binary labels 0 and 1")
    if not (np.all(np.isfinite(A_arr)) and np.all(np.isfinite(y_arr))):
        raise ValueError("Inputs must contain only finite numbers")
    if eta <= 0 or not np.isfinite(eta):
        raise ValueError("Learning rate eta must be positive and finite")
    if n_iterations < 1:
        raise ValueError("Number of iterations must be at least 1")

    theta = np.zeros(2, dtype=np.float64) if theta0 is None else np.asarray(theta0, dtype=np.float64).copy()
    
    history = {
        "theta": np.zeros((n_iterations + 1, 2), dtype=np.float64),
        "loss": np.zeros(n_iterations + 1, dtype=np.float64),
        "gradient": np.zeros((n_iterations + 1, 2), dtype=np.float64),
        "accuracy": np.zeros(n_iterations + 1, dtype=np.float64),
        "boundary": np.zeros(n_iterations + 1, dtype=np.float64),
    }

    for k in range(n_iterations + 1):
        z = A_arr @ theta
        p = sigmoid(z)
        loss = cross_entropy_loss(y_arr, p)
        grad = cross_entropy_gradient(A_arr, y_arr, p)
        y_hat = (p >= 0.5).astype(int)
        acc = float(np.mean(y_hat == y_arr))
        boundary = -theta[0] / theta[1] if theta[1] != 0.0 else np.nan

        history["theta"][k] = theta
        history["loss"][k] = loss
        history["gradient"][k] = grad
        history["accuracy"][k] = acc
        history["boundary"][k] = boundary

        if k < n_iterations:
            theta = theta - eta * grad

    return history

# Execute 10 iterations of gradient descent
history_10 = train_logistic_gd(A, y, eta=0.1, n_iterations=10)

print(f"{'State':>5} | {'b':>10} | {'w':>10} | {'boundary':>10} | {'Loss':>10} | {'grad_b':>10} | {'grad_w':>10} | {'acc':>6}")
print("-" * 88)
for k in range(11):
    b_val = history_10["theta"][k, 0]
    w_val = history_10["theta"][k, 1]
    bound_val = history_10["boundary"][k]
    loss_val = history_10["loss"][k]
    gb_val = history_10["gradient"][k, 0]
    gw_val = history_10["gradient"][k, 1]
    acc_val = history_10["accuracy"][k]
    bound_str = f"{bound_val:10.6f}" if np.isfinite(bound_val) else f"{'—':>10}"
    print(f"{k:5d} | {b_val:10.6f} | {w_val:10.6f} | {bound_str} | {loss_val:10.6f} | {gb_val:10.6f} | {gw_val:10.6f} | {acc_val:6.3f}")

State |          b |          w |   boundary |       Loss |     grad_b |     grad_w |    acc

----------------------------------------------------------------------------------------

    0 |   0.000000 |   0.000000 |          — |   0.693147 |   0.125000 |  -0.875000 |  0.375

    1 |  -0.012500 |   0.087500 |   0.142857 |   0.620153 |   0.132734 |  -0.756989 |  0.875

    2 |  -0.025773 |   0.163199 |   0.157927 |   0.564805 |   0.138485 |  -0.658488 |  0.875

    3 |  -0.039622 |   0.229048 |   0.172985 |   0.522192 |   0.142526 |  -0.577482 |  0.875

    4 |  -0.053874 |   0.286796 |   0.187850 |   0.488733 |   0.145227 |  -0.510994 |  0.875

    5 |  -0.068397 |   0.337895 |   0.202421 |   0.461917 |   0.146923 |  -0.456149 |  0.875

    6 |  -0.083089 |   0.383510 |   0.216655 |   0.439996 |   0.147879 |  -0.410525 |  0.875

    7 |  -0.097877 |   0.424563 |   0.230537 |   0.421748 |   0.148294 |  -0.372205 |  0.875

    8 |  -0.112707 |   0.461783 |   0.244069 |   0.406306 |   0.

In [3]:
# S7 — Verification of Fixed Teaching Calculations
RTOL = 0.0
ATOL = 1e-12

# 1. Shapes and dtypes
assert x.shape == (8,) and y.shape == (8,) and A.shape == (8, 2)
assert x.dtype == np.float64 and y.dtype == np.float64 and A.dtype == np.float64

# 2. Sigmoid properties and derivative
assert sigmoid(0.0) == 0.5
grid_z = np.linspace(-5.0, 5.0, 101)
p_grid = sigmoid(grid_z)
assert np.all((p_grid > 0.0) & (p_grid < 1.0))
eps_diff = 1e-6
num_deriv = (sigmoid(grid_z + eps_diff) - sigmoid(grid_z - eps_diff)) / (2 * eps_diff)
analytical_deriv = p_grid * (1.0 - p_grid)
np.testing.assert_allclose(num_deriv, analytical_deriv, rtol=1e-5, atol=1e-5)

# 3. State 0 exact identities
np.testing.assert_allclose(history_10["loss"][0], np.log(2.0), rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(history_10["gradient"][0, 0], 0.125, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(history_10["gradient"][0, 1], -0.875, rtol=RTOL, atol=ATOL)
assert history_10["accuracy"][0] == 0.375

# 4. State 1 invariant boundary (1/7)
np.testing.assert_allclose(history_10["boundary"][1], 1.0 / 7.0, rtol=RTOL, atol=ATOL)
np.testing.assert_allclose(history_10["theta"][1], [-0.0125, 0.0875], rtol=RTOL, atol=ATOL)
assert history_10["accuracy"][1] == 0.875

# 5. State 10 match against Table T1 references
expected_theta_10 = np.array([-0.142342, 0.526940], dtype=np.float64)
expected_loss_10 = 0.381510
expected_grad_10 = np.array([0.147547, -0.287826], dtype=np.float64)
np.testing.assert_allclose(history_10["theta"][10], expected_theta_10, rtol=RTOL, atol=1e-5)
np.testing.assert_allclose(history_10["loss"][10], expected_loss_10, rtol=RTOL, atol=1e-5)
np.testing.assert_allclose(history_10["gradient"][10], expected_grad_10, rtol=RTOL, atol=1e-5)

# 6. Monotonicity of loss
for k in range(10):
    assert history_10["loss"][k + 1] < history_10["loss"][k]

# 7. Long-run milestone k* = 139
history_150 = train_logistic_gd(A, y, eta=0.1, n_iterations=140)
assert history_150["accuracy"][138] == 0.875
assert history_150["accuracy"][139] == 1.000
assert history_150["boundary"][138] < 1.0
assert history_150["boundary"][139] > 1.0

# 8. Equivalence of elementwise summation and matrix gradient form
p_state0 = sigmoid(A @ history_10["theta"][0])
res_sum = np.sum(p_state0 - y) / N
res_x_sum = np.sum((p_state0 - y) * x) / N
np.testing.assert_allclose(history_10["gradient"][0], [res_sum, res_x_sum], rtol=RTOL, atol=ATOL)

# 9. Input validation error raising
try:
    train_logistic_gd(A, y, eta=-0.1, n_iterations=10)
    assert False, "Should have raised ValueError for negative eta"
except ValueError as e:
    assert "eta must be positive" in str(e)

print("PASS: all 03.02 fixed teaching calculations verified.")

PASS: all 03.02 fixed teaching calculations verified.



In [4]:
# S8 — Independent Cross-Checks: Pure Python (Pathway A)
import math

def pure_sigmoid(val: float) -> float:
    return 1.0 / (1.0 + math.exp(-val))

pure_theta = [0.0, 0.0]
x_list = [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0]
y_list = [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0]

for step in range(10):
    grad_b = 0.0
    grad_w = 0.0
    for xi, yi in zip(x_list, y_list):
        zi = pure_theta[0] + pure_theta[1] * xi
        pi = pure_sigmoid(zi)
        diff = pi - yi
        grad_b += diff
        grad_w += diff * xi
    pure_theta[0] -= 0.1 * (grad_b / 8.0)
    pure_theta[1] -= 0.1 * (grad_w / 8.0)

np.testing.assert_allclose(pure_theta, history_10["theta"][10], rtol=RTOL, atol=1e-12)
print("PASS: Pathway A (pure-Python math) matches NumPy GD to 1e-12.")

PASS: Pathway A (pure-Python math) matches NumPy GD to 1e-12.



### Pathway B: scikit-learn Comparison
We cross-check our implementation against scikit-learn's standard `LogisticRegression` estimator.

> **Important Conceptual Note:** scikit-learn's default formulation differs from our teaching implementation in two crucial ways:
> 1. **L2 Regularization by default:** scikit-learn defaults to `penalty='l2'` with inverse regularisation strength $C = 1.0$.
> 2. **Sum-form objective:** scikit-learn minimises the unnormalized sum $\frac{1}{2}\|w\|^2 + C \sum_{i=1}^N \ell_i$, rather than the mean loss $\frac{1}{N}\sum \ell_i$.
> 
> When trained without regularization (`penalty=None`), scikit-learn encounters the same fundamental phenomenon of linear separability: the weights grow large ($\|\theta\| > 20$) while the decision boundary stabilizes at $x^* \approx 1.472$, cleanly within the margin $(1.0, 2.0)$.

---

In [5]:
from sklearn.linear_model import LogisticRegression

# 1. Default scikit-learn (C=1.0, L2 regularized, L-BFGS solver)
clf_default = LogisticRegression(C=1.0, penalty="l2", solver="lbfgs")
clf_default.fit(x.reshape(-1, 1), y)
b_sk = float(clf_default.intercept_[0])
w_sk = float(clf_default.coef_[0, 0])
boundary_sk = -b_sk / w_sk

print(f"sklearn default (C=1.0): b={b_sk:.6f}, w={w_sk:.6f}, boundary={boundary_sk:.6f}")

# 2. Unregularized scikit-learn (penalty=None)
clf_none = LogisticRegression(penalty=None, solver="lbfgs")
clf_none.fit(x.reshape(-1, 1), y)
b_none = float(clf_none.intercept_[0])
w_none = float(clf_none.coef_[0, 0])
boundary_none = -b_none / w_none

print(f"sklearn unregularized:   b={b_none:.6f}, w={w_none:.6f}, boundary={boundary_none:.6f}")
print("Both estimators confirm boundary sits strictly in separating margin (1.0, 2.0).")

sklearn default (C=1.0): b=-1.760949, w=1.155912, boundary=1.523428

sklearn unregularized:   b=-21.558119, w=14.646218, boundary=1.471924

Both estimators confirm boundary sits strictly in separating margin (1.0, 2.0).



## S9 — Visual Explanations & Diagnostics

We inspect two static visualizations capturing the optimization dynamics.

### Visual 1 (V1) — Loss and Accuracy Trace Across 5,000 Iterations
![Visual 1: Loss and Accuracy Trace](assets/loss_accuracy_trace.png)

**Caption:** Two-panel trace of full-batch gradient descent across 5,000 iterations on eight synthetic points. The top panel shows mean cross-entropy loss (blue), which falls smoothly from $L(0) = 0.6931 \to L(5000) = 0.0221$ without ever reaching zero. The bottom panel shows classification accuracy (vermillion), which exhibits a discrete step function: $0.375$ at State 0, jumping to $0.875$ at State 1, and stepping to $1.0$ at iteration $k^* = 139$ when the decision boundary crosses $x=1.0$.

**Text Description (Accessibility Alt-Text):** Two stacked line charts sharing a common horizontal axis of iteration $k$ from 0 to 5,000. In the upper panel, cross-entropy loss begins at 0.6931 at iteration 0, drops steeply to below 0.1 by iteration 500, and gently slopes downward to 0.0221 at iteration 5,000. A shaded vertical bar marks the hand-trace region of iterations 0 to 10. In the lower panel, accuracy begins at 0.375 (3/8), instantly steps to 0.875 (7/8) at iteration 1, remains completely horizontal until iteration 139, where a marker highlights the step to 1.0 (8/8), remaining at 1.0 through iteration 5,000.

---

### Visual 2 (V2) — Decision Boundary Evolution in Feature Space
![Visual 2: Decision Boundary Evolution](assets/decision_boundary_evolution.png)

**Caption:** Evolution of the logistic curve $p(x) = \sigma(b + wx)$ in feature space. At State 0 (grey dotted), the model is flat at $p \equiv 0.5$. By State 10 (blue solid), the curve has formed a soft S-curve with decision boundary $x^* = 0.270$. By iteration 5,000 (vermillion solid), the curve has sharpened into a steep, near-step separator with boundary $x^* = 1.446$, situated directly in the gap between the negative class ($x \le 1.0$, sky blue circles) and the positive class ($x \ge 2.0$, vermillion squares).

**Text Description (Accessibility Alt-Text):** A single-panel plot with feature $x$ on the horizontal axis from -4.0 to 5.0 and predicted probability $p$ on the vertical axis from 0.0 to 1.0. Five negative class points are plotted as sky-blue circles at $y=0$ for $x \in \{-3, -2, -1, 0, 1\}$. Three positive class points are plotted as vermillion squares at $y=1$ for $x \in \{2, 3, 4\}$. A horizontal dash-dot line marks the decision threshold at $p = 0.5$. Three logistic curves are drawn: a horizontal line at $p=0.5$ representing State 0; a gentle blue S-curve representing State 10 with a vertical dashed line marking its boundary at $x=0.27$; and a very steep red S-curve representing iteration 5,000 with a vertical dashed line marking its boundary at $x=1.45$.

---

## S10 — Failure Modes, Misconceptions & Spot the Bug

### 21 Critical Misconceptions Dispelled
1. **"Cross-entropy and MSE are interchangeable for classification."**  
   *False.* Composing MSE with the sigmoid produces non-convex landscapes and severe gradient saturation when predictions are confidently wrong.
2. **"Cross-entropy can be negative."**  
   *False.* Because probabilities $p_i \in (0, 1)$, $-\ln(p_i) > 0$. Cross-entropy is strictly non-negative ($L \ge 0$).
3. **"Lower loss always guarantees higher accuracy."**  
   *False.* Across States 1 to 10, the loss falls from $0.620 \to 0.381$, but accuracy is frozen at $0.875$.
4. **"Higher accuracy always guarantees lower loss."**  
   *False.* A model with $87.5\%$ accuracy that predicts probabilities with timid confidence ($p = 0.51$) can have a higher loss than a model with the same accuracy that predicts with strong confidence ($p = 0.95$).
5. **"The gradient descent update for classification uses the same formula as regression."**  
   *Distinction:* The update *rule* $\theta \leftarrow \theta - \eta \nabla L$ is the same. The *gradient* substituted into it is different ($\frac{1}{N}A^T(p - y)$ vs $\frac{2}{N}A^T(\hat{y} - y)$).
6. **"$\sigma'(z) = 1/\sigma(z)$."**  
   *False.* The derivative of the sigmoid is $\sigma(z)(1 - \sigma(z))$.
7. **"Because $\sigma'$ cancels out in the derivation, it is irrelevant."**  
   *False.* The cancellation is the entire mathematical explanation of why cross-entropy prevents gradient saturation.
8. **"A loss of zero is achievable on real classification datasets."**  
   *False.* Loss equals zero only when $p_i \in \{0.0, 1.0\}$, which requires infinite linear scores ($|z| \to \infty$).
9. **"Gradient descent always converges in a fixed number of steps."**  
   *False.* On separable data, unregularized gradient descent diverges in norm indefinitely.
10. **"A learning rate that works for linear regression works for logistic regression."**  
    *False.* Suitable learning rates depend on the Lipschitz constant (spectral norm of $A^T A$), not just the model family.
11. **"The gradient formula $\frac{1}{N}A^T(p - y)$ holds for any classification loss."**  
    *False.* It is unique to cross-entropy composed with the logistic sigmoid.
12. **"Logistic regression has a closed-form normal equation like OLS."**  
    *False.* The non-linear sigmoid prevents an analytical closed-form solution. Iterative optimization is mandatory.
13. **"Cross-entropy is the only valid classification loss."**  
    *False.* Hinge loss (SVMs) and exponential loss (AdaBoost) are valid alternatives with different inductive biases.
14. **"Cross-entropy uses base-10 logarithms."**  
    *False.* Cross-entropy uses natural logarithms ($\ln$, base $e$). Using $\log_{10}$ scales the loss by $\ln(10) \approx 2.3026$.
15. **"If accuracy doesn't change after a step, the step was wasted."**  
    *False.* The step moves predicted probabilities toward greater confidence and decreases the loss.
16. **"The coefficients after 10 steps are optimal."**  
    *False.* On separable data, any finite iterate is merely a snapshot on a diverging ray.
17. **"Linear separability guarantees a unique finite maximum likelihood estimate."**  
    *False.* Separability causes the unregularized MLE to fail to exist at any finite $\theta$ (Albert & Anderson, 1984; Soudry et al., 2018).
18. **"Clipping probabilities to $[10^{-15}, 1 - 10^{-15}]$ changes the math."**  
    *False.* Clipping is purely a numerical guardrail against $\ln(0) = -\infty$ in floating-point arithmetic.
19. **"Regularization is required for gradient descent to step."**  
    *False.* Unregularized GD runs smoothly; its parameters simply grow in magnitude while their ratio converges.
20. **"The discrepancy $p - y$ is a linear regression residual."**  
    *Distinction:* It has identical algebraic structure, but $p \in (0, 1)$ is a bounded probability, not an unbounded prediction.
21. **"Decreasing training cross-entropy guarantees generalization."**  
    *False.* Minimizing training loss on separable data pushes weights to infinity, which can lead to severe overconfidence and overfitting on unseen test data.

---

### Spot the Bug
Examine this student implementation of the logistic regression gradient:

```python
def buggy_logistic_gradient(A, y, theta):
    z = A @ theta
    p = 1.0 / (1.0 + np.exp(-z))
    # Buggy gradient computation:
    grad = (A.T @ (p - y)) * np.log10(np.e)
    return grad
```

**Diagnose the bugs:**
1. *Bug 1 (Missing $\frac{1}{N}$ scaling):* The sum over examples is not divided by $N$. As the dataset grows, the gradient norm scales linearly with $N$, destabilizing gradient descent.
2. *Bug 2 (Incorrect Log Base Scaling):* Multiplying by $\log_{10}(e)$ implies the loss used base-10 logarithms, but the sigmoid cancellation strictly requires natural logarithms.

---

## S11 — Graded Exercises

### E1 (Conceptual — The $-\ln(p)$ Penalty Profile)
Compute the loss contribution $\ell_i$ for a positive example ($y_i = 1$) when the predicted probability is:
(a) $p = 0.999$, (b) $p = 0.500$, (c) $p = 0.010$, (d) $p = 10^{-6}$.

<details>
<summary>Click to view solution</summary>

For $y = 1$, $\ell = -\ln(p)$:
- (a) $-\ln(0.999) \approx 0.001001$ (nearly zero penalty).
- (b) $-\ln(0.500) = \ln 2 \approx 0.693147$.
- (c) $-\ln(0.010) \approx 4.605170$.
- (d) $-\ln(10^{-6}) \approx 13.815511$.
Notice that moving from $0.50 \to 0.999$ decreases loss by $\approx 0.69$, but dropping from $0.50 \to 0.01$ increases loss by almost $4.0$ units. Confident mistakes are punished severely.
</details>

---

### E2 (Hand Calculation — Alternate Learning Rate $\eta = 0.2$)
Recompute the first parameter update from State 0 using $\eta = 0.2$ instead of $\eta = 0.1$.
Find $b_1, w_1$, and the decision boundary location $x^* = -b_1/w_1$.

<details>
<summary>Click to view solution</summary>

From State 0: $\nabla_b L_0 = 0.125, \nabla_w L_0 = -0.875$.
$$b_1 = 0 - 0.2(0.125) = -0.025000$$
$$w_1 = 0 - 0.2(-0.875) = +0.175000$$
Decision boundary:
$$x^* = -\frac{b_1}{w_1} = -\frac{-0.025}{0.175} = \frac{25}{175} = \frac{1}{7} \approx 0.142857$$
The decision boundary after step 1 is *identical* to the one obtained with $\eta = 0.1$. The ratio $-b_1/w_1 = -(-\eta \nabla_b)/(\eta (-\nabla_w)) = -\nabla_b / \nabla_w$ cancels out $\eta$ entirely!
</details>

---

### E3 (Conceptual — Why Accuracy Stays Flat Across States 1 to 10)
Examine Table T3 (State 1) and Table T4 (State 2). Prove why the discrete prediction $\hat{y}_i$ for every example is identical in State 1 and State 2.

<details>
<summary>Click to view solution</summary>

Under threshold $0.5$, $\hat{y}_i = 1 \iff z_i \ge 0 \iff x_i \ge x^* = -b/w$.
In State 1, the boundary is $x^* \approx 0.143$. In State 2, the boundary is $x^* \approx 0.158$.
Both boundaries sit strictly inside the interval between example 4 ($x=0.0$) and example 5 ($x=1.0$).
Because no training example lies between $0.143$ and $0.158$, every example remains on the exact same side of the decision boundary. Thus $\hat{y}_i$ does not change for any point, and accuracy remains exactly $7/8 = 0.875$.
</details>

---

### E4 (Implementation — Early Stopping on Training Accuracy)
Write a Python function `train_until_accuracy(A, y, target_acc=1.0)` that runs gradient descent until the training accuracy reaches $100\%$, returning the stopping iteration $k^*$. Verify that $k^* = 139$.

---

### E5 (Implementation — Plotting the Cross-Entropy Contour Map)
Using `matplotlib`, evaluate and plot contour lines of $L(b, w)$ over the box $(b, w) \in [-4, 2] \times [-1, 3]$. Overlay the first 10 steps of the gradient descent path from Table T1.

---

### E6 (Extension — Non-Separable Overlapping Data)
Modify the dataset by swapping the labels of $x = 1.0$ ($y=1$) and $x = 2.0$ ($y=0$), restoring the overlapping data from Lesson 03.01. Run gradient descent for 5,000 iterations. Verify that with overlapping data, gradient descent converges to a finite optimum $\theta^* \approx [-1.923440, 1.282293]$ with non-zero minimum loss $L^* \approx 0.312881$.

---

## S12 — Takeaways, Continuity & References

### Summary of Core Principles
1. **The Logistic Regression Pipeline:**
   $$\text{Features } x_i \xrightarrow{\text{linear}} z_i = b + wx_i \xrightarrow{\text{sigmoid}} p_i = \sigma(z_i) \xrightarrow{\text{cross-entropy}} L(\theta) \xrightarrow{\text{GD}} \theta \leftarrow \theta - \frac{\eta}{N}A^T(p - y)$$
2. **The Great Cancellation:** Differentiating binary cross-entropy cancels the $p(1-p)$ term from the sigmoid derivative, producing a gradient proportional to the linear discrepancy $p - y$.
3. **Objective vs. Metric:** Cross-entropy is a smooth, differentiable surface that guides parameter updates; accuracy is a piecewise-constant step function used solely for evaluation.
4. **The Separability Reality:** On linearly separable data, unregularized logistic regression has no finite optimum. The weights grow infinitely large while the decision boundary stabilizes at the max-margin separator.

---

### Continuity & What Comes Next
- **Closing the Loop with 03.01:** In 03.01, we evaluated a fixed model with $b = -1.5, w = 1.0$. Now we know that gradient descent on separable data drifts toward that exact boundary ($x^* = 1.5$), and those hand-picked values were simply a snapshot on the diverging ray.
- **Looking Ahead to 03.03 (K-Nearest Neighbors):** Logistic regression assumes a linear boundary $b + w^T x = 0$. What if the decision boundary is non-linear or multi-modal? Lesson 03.03 introduces non-parametric classification.
- **Looking Ahead to 03.08–03.10 (Support Vector Machines):** Soudry et al. (2018) proved that unregularized logistic regression on separable data converges in direction to the hard-margin SVM solution. In 03.08, we will study Support Vector Machines, which maximize that margin directly in a single optimization problem.
- **Looking Ahead to Module 07 (Evaluation):** Accuracy alone can be misleading when classes are imbalanced. Module 07 explores Precision, Recall, and ROC/AUC curves.

---

### Authoritative References
1. **Goodfellow, I., Bengio, Y., & Courville, A. (2016).** *Deep Learning*, MIT Press.
   - §6.2.1.2: Saturation effects in neural network outputs.
   - §6.2.2.2: Cross-entropy and the sigmoid exponential cancellation.
2. **Soudry, D., Hoffer, E., Nacson, M. S., Gunasekar, S., & Srebro, N. (2018).** The Implicit Bias of Gradient Descent on Separable Data. *Journal of Machine Learning Research (JMLR)*, 19(70):1–57.
3. **Bishop, C. M. (2006).** *Pattern Recognition and Machine Learning*, Springer.
   - §4.3.2: Logistic regression, cross-entropy error function, and gradient descent.
4. **Hastie, T., Tibshirani, R., & Friedman, J. (2009).** *The Elements of Statistical Learning* (2nd ed.), Springer.
   - Chapter 4: Linear Methods for Classification.
5. **scikit-learn Documentation (v1.5).** `sklearn.linear_model.LogisticRegression`.
6. **NumPy Documentation (v1.26).** `numpy.log`, `numpy.exp`.